# Volume 06 — Gradient Boosting & Hyperparameter Optimization
### XGBoost, LightGBM, CatBoost, and Optuna — The Tools That Win Tabular Business Problems

*Consolidated from: Grok Chapter 20, ChatGPT Chapters 23–25 & 28 (LightGBM, XGBoost, CatBoost, Optuna)*

---

## Why This Volume Matters

Here is a fact that surprises people who associate "AI" with deep learning and neural networks: for **structured, tabular business data** — customer records, transaction logs, spreadsheet-shaped data with rows and named columns, which is the overwhelming majority of what a business analyst actually works with — **gradient-boosted trees consistently outperform deep learning**, are faster to train, need far less data, and are dramatically easier to tune and deploy. If you take away one practical fact from this entire course about "which model should I actually reach for at work," it's this: for tabular business problems, start with a gradient boosting library, not a neural network.

### The Core Idea: Gradient Boosting in Plain Language

Volume 05 introduced a single decision tree. **Gradient boosting** builds an *ensemble* — hundreds of small, deliberately weak decision trees — one at a time, where each new tree is trained specifically to correct the mistakes of all the trees built before it. The final prediction is the combined (weighted) output of every tree in the sequence. This "learn from your own errors, repeatedly" process is why boosted trees are so consistently strong on tabular data: they progressively hunt down and correct whatever pattern the model has gotten wrong so far, rather than trying to get everything right in one shot.

Three libraries dominate this space in industry, and the difference between them is largely about *speed*, *categorical feature handling*, and *ecosystem maturity* rather than fundamentally different modeling philosophy:

| Library | Signature Strength | Best For |
|---|---|---|
| **XGBoost** | Battle-tested, excellent defaults, huge industry adoption, strong regularization | The safe, well-documented default choice |
| **LightGBM** | Extremely fast, low memory, leaf-wise tree growth | Large datasets, fast iteration |
| **CatBoost** | Best native categorical feature handling, strong defaults out of the box | Data with many text/categorical columns (region, plan, channel) |

### A Shared Realistic Dataset for This Volume

All three libraries are demonstrated on the same synthetic churn dataset, so you can directly compare their outputs like-for-like — exactly as a real model-selection exercise at work would:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

np.random.seed(42)
n = 5000
df = pd.DataFrame({
    "tenure": np.random.randint(1, 72, n),
    "monthly_charges": np.round(np.random.uniform(20, 120, n), 2),
    "contract": np.random.choice(["Month-to-month", "One year", "Two year"], n, p=[0.55, 0.25, 0.2]),
    "internet": np.random.choice(["DSL", "Fiber", "None"], n, p=[0.35, 0.45, 0.2]),
    "tech_support": np.random.choice(["Yes", "No"], n, p=[0.35, 0.65]),
    "senior": np.random.binomial(1, 0.15, n),
    "num_tickets": np.random.poisson(1.2, n)
})

logit = (-1.8 + 0.02*df["monthly_charges"] - 0.035*df["tenure"]
         + 0.9*(df["contract"] == "Month-to-month").astype(int)
         + 0.45*(df["tech_support"] == "No").astype(int)
         + 0.25*df["num_tickets"] + 0.4*df["senior"])
df["churn"] = (np.random.rand(n) < 1/(1+np.exp(-logit))).astype(int)

print(df["churn"].value_counts(normalize=True).round(3))

Note the class imbalance printed here — this is deliberate and realistic (churn datasets are almost never 50/50), and it's exactly why ROC-AUC (Volume 05) rather than raw accuracy is used to evaluate every model in this volume.

---